<a href="https://colab.research.google.com/github/Segn11/datasciencebootcamp_project/blob/showcase/challenge_zindi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv('/content/Train (1).csv')
test = pd.read_csv('/content/Test (1).csv')



In [ ]:
import pandas as pd
import numpy as np
# Load data
train = pd.read_csv('Train (1).csv')
test = pd.read_csv('Test (1).csv')

In [ ]:
# Separate the numeric columns from non-numeric ones
numeric_columns = train.select_dtypes(include=['float64', 'int64']).columns
non_numeric_columns = train.select_dtypes(exclude=['float64', 'int64']).columns

# Fill missing values for numeric columns with the mean
train[numeric_columns] = train[numeric_columns].fillna(train[numeric_columns].mean())

# For non-numeric columns, fill missing values with the mode (or other strategies you prefer)
for column in non_numeric_columns:
    train[column] = train[column].fillna(train[column].mode()[0])

# Now check the filled data
print(train.isnull().sum())
///


def preprocess(df):
    df['Date'] = pd.to_datetime(df['Date'])

    # Extract datetime features
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day_of_year'] = df['Date'].dt.dayofyear
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)  # Week of the year
    df['day_of_week'] = df['Date'].dt.dayofweek  # 0: Monday, 6: Sunday
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)  # 1 if weekend, 0 if weekday

    # Week of the month (1-4), using the day of the month to estimate it
    df['week_of_month'] = (df['Date'].dt.day - 1) // 7 + 1  # Week of the month (1-4)

    # Categorical Encoding: Target Encoding for Place_ID
    # Calculate the mean target for each Place_ID (if it exists in the dataset)
    if 'target' in df.columns:
        place_target_mean = df.groupby('Place_ID')['target'].mean()
        df['place_id_encoded'] = df['Place_ID'].map(place_target_mean)

    # Drop the original Date column after extracting useful features
    df.drop(['Date'], axis=1, inplace=True)

    return df
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import numpy as np

# Prepare datasets
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_val, label=y_val)

# Parameters for the model
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'max_depth': 7
}

# Create callbacks for early stopping and logging
callbacks = [
    lgb.early_stopping(50),  # Stop after 50 rounds of no improvement
    lgb.log_evaluation(100)  # Log evaluation every 100 rounds
]

# Train the model using the train method with early stopping and logging
model = lgb.train(
    params=params,
    train_set=train_data,
    valid_sets=[train_data, valid_data],
    callbacks=callbacks,
    num_boost_round=1000
)

# Make predictions
y_pred = model.predict(X_val, num_iteration=model.best_iteration)

# Evaluate the model using RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print(f'RMSE: {rmse}')
# Make predictions on the test set
test_pred = model.predict(X_test, num_iteration=model.best_iteration)

# Create submission dataframe
submission = pd.read_csv('Test (1).csv')[['Place_ID X Date']]
submission['target'] = test_pred

# Save submission
submission.to_csv('submission.csv', index=False)

